In [ ]:
from transformers import BartTokenizer
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from Utils import TimexNorm_Utils
from Reader import obtain_combined_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

In [ ]:
tokenizer.add_special_tokens({"additional_special_tokens": ["<timex","type=DATE>","type=TIME>","type=DURATION>","type=SET>","</timex>", "<sep>"]})
model.resize_token_embeddings(len(tokenizer))

In [ ]:
datasets = obtain_combined_dataset(["TempEval3","wikiwars","tweets"], "normalised")

In [ ]:
datasets = utils.tokenize_datasets(datasets)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    predict_with_generate=True,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    compute_metrics=utils.compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

In [ ]:
trainer.evaluate(datasets["test"])